# Intent-Aware RAG Question Answering System

This notebook implements the project described in the resume:

1. Classify a question into one of 10 intent classes using Sentence Transformers + Logistic Regression.
2. Retrieve relevant CSE knowledge passages using dense embeddings + FAISS.
3. Check for potentially unsafe requests using BART-large-MNLI zero-shot classification.
4. Generate a concise, context-grounded answer using FLAN-T5.

**Input files expected in the notebook working directory**
- `train_data.csv`: columns `question`, `intent`
- `test.csv`: columns `question`, `intent`
- `cse_topics.csv`: column `text` (optionally an `id` column)

The intent labels should match the labels in `INTENT_MAP` below. The reported 90.6% accuracy is a result to reproduce on your actual held-out test set; this notebook does not assume or fabricate that score.


## 1. Install dependencies

In [ ]:
%pip install -q pandas numpy scikit-learn sentence-transformers faiss-cpu transformers torch sentencepiece accelerate

## 2. Imports and configuration

In [ ]:
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
import torch

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from transformers import pipeline

def find_data_file(filename):
    for p in [Path(filename), Path('data') / filename, Path('..') / 'data' / filename]:
        if p.exists():
            return str(p)
    return filename

DEVICE = 0 if torch.cuda.is_available() else -1

INTENT_MAP = {
    "ABBR": "Abbreviation",
    "ENTY": "Entity",
    "HUM": "Human",
    "LOC": "Location",
    "NUM": "Number",
    "DESC:definition": "Definition",
    "DESC:procedure": "Procedure",
    "DESC:reason": "Reason",
    "DESC:comparison": "Comparison",
    "DESC:description": "Description",
}

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


## 3. Load intent training and test data

In [ ]:
train_df = pd.read_csv(find_data_file('train_data.csv'))
test_df = pd.read_csv(find_data_file('test.csv'))

required_columns = {"question", "intent"}
assert required_columns.issubset(train_df.columns), "train_data.csv needs question and intent columns"
assert required_columns.issubset(test_df.columns), "test.csv needs question and intent columns"

X_train = train_df["question"].fillna("").map(clean_text).tolist()
y_train = train_df["intent"].astype(str).tolist()
X_test = test_df["question"].fillna("").map(clean_text).tolist()
y_test = test_df["intent"].astype(str).tolist()

print("Training examples:", len(X_train))
print("Test examples:", len(X_test))
print("Intent classes:", sorted(set(y_train)))


## 4. Convert questions into sentence embeddings

In [ ]:
# all-MiniLM-L6-v2 converts each question into a dense 384-dimensional vector.
intent_encoder = SentenceTransformer("all-MiniLM-L6-v2")

X_train_emb = intent_encoder.encode(
    X_train, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=True
)
X_test_emb = intent_encoder.encode(
    X_test, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding matrix:", X_train_emb.shape)


## 5. Train and evaluate the intent classifier

In [ ]:
intent_model = LogisticRegression(max_iter=2000)
intent_model.fit(X_train_emb, y_train)

test_predictions = intent_model.predict(X_test_emb)
print("Intent accuracy:", accuracy_score(y_test, test_predictions))
print(classification_report(y_test, test_predictions, zero_division=0))

def predict_intent(question):
    question = clean_text(question)
    embedding = intent_encoder.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True
    )
    return intent_model.predict(embedding)[0]

def predict_project_intent(question):
    raw_label = predict_intent(question)
    return INTENT_MAP.get(raw_label, raw_label)


## 6. Load the knowledge base

In [ ]:
knowledge_df = pd.read_csv(find_data_file('cse_topics.csv'))

if "text" not in knowledge_df.columns:
    raise ValueError("cse_topics.csv must contain a 'text' column")

documents = knowledge_df["text"].dropna().astype(str).tolist()
documents = [doc.strip() for doc in documents if doc.strip()]

if not documents:
    raise ValueError("Knowledge base is empty")

print("Knowledge passages:", len(documents))
print("Example passage:", documents[0][:300])


## 7. Embed documents and build the FAISS index

In [ ]:
# Normalize document and query vectors, then use inner product as cosine similarity.
retriever = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = retriever.encode(
    documents,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

embedding_dimension = doc_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dimension)
faiss_index.add(doc_embeddings)

print("FAISS vectors stored:", faiss_index.ntotal)

def retrieve_docs(question, k=3):
    query_embedding = retriever.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")

    k = min(k, len(documents))
    scores, indices = faiss_index.search(query_embedding, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            results.append({
                "text": documents[idx],
                "similarity": float(score)
            })
    return results


## 8. Load FLAN-T5 for grounded answer generation

In [ ]:
# FLAN-T5 is a text-to-text model, so use the text2text-generation pipeline.
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=DEVICE
)

def generate_answer(question, intent, retrieved_docs):
    if not retrieved_docs:
        return "I could not find relevant information in the knowledge base."

    context = "\\n\\n".join(
        item["text"] for item in retrieved_docs[:3]
    )

    prompt = f\'''You are a question-answering assistant.
Answer the user's question using only the supplied context.
The detected question intent is: {intent}.
Use that intent to shape the answer (for example, explain a definition,
give steps for a procedure, or state a reason).
If the context does not contain the answer, say that the available
documents do not provide enough information.
Keep the answer concise (under 100 words).

Context:
{context}

Question: {question}

Answer:
\'''

    output = generator(
        prompt,
        max_new_tokens=120,
        do_sample=False,
        truncation=True
    )
    return output[0]["generated_text"].strip()


## 9. Zero-shot safety screening

In [ ]:
# This is a lightweight screening layer, not a complete safety system.
safety_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=DEVICE
)

RISK_LABELS = [
    "safe and benign request",
    "self-harm instructions or encouragement",
    "instructions for illegal activity",
    "request to harm people or facilitate violence",
    "dangerous weapon or explosive construction instructions"
]

def detect_query_risk(question, risk_threshold=0.65):
    result = safety_classifier(
        question,
        candidate_labels=RISK_LABELS,
        multi_label=False
    )

    top_label = result["labels"][0]
    top_score = float(result["scores"][0])

    is_risky = (
        top_label != "safe and benign request"
        and top_score >= risk_threshold
    )
    return {
        "is_risky": is_risky,
        "label": top_label,
        "score": top_score
    }

def unsafe_message():
    return (
        "I can’t help with instructions that could enable harm. "
        "If your goal is safety or prevention, I can help with "
        "general precautions and reliable safety information."
    )


## 10. Connect all components into one RAG pipeline

In [ ]:
def ask_system(question, top_k=3):
    # Step 1: understand the question
    intent = predict_project_intent(question)

    # Step 2: screen the query before retrieval and generation
    safety_result = detect_query_risk(question)
    if safety_result["is_risky"]:
        return {
            "intent": intent,
            "blocked": True,
            "safety": safety_result,
            "retrieved_documents": [],
            "answer": unsafe_message()
        }

    # Step 3: retrieve relevant knowledge
    retrieved = retrieve_docs(question, k=top_k)

    # Step 4: generate an answer grounded in retrieved passages
    answer = generate_answer(question, intent, retrieved)

    return {
        "intent": intent,
        "blocked": False,
        "safety": safety_result,
        "retrieved_documents": retrieved,
        "answer": answer
    }


## 11. Try example questions

In [ ]:
questions = [
    "What is supervised learning?",
    "Why do we use normalization in databases?",
    "Compare TCP and UDP.",
    "How does binary search work?"
]

for question in questions:
    result = ask_system(question)
    print("\\nQuestion:", question)
    print("Intent:", result["intent"])
    print("Answer:", result["answer"])
    print("Retrieved passages:")
    for item in result["retrieved_documents"]:
        print(f"- (similarity={item['similarity']:.3f}) {item['text'][:180]}")


## 12. What each metric/component means

- **Intent accuracy:** fraction of held-out questions whose intent label was predicted correctly.
- **Sentence embedding:** a vector representation of a sentence; semantically similar questions tend to have nearby vectors.
- **FAISS:** indexes vectors and quickly finds the closest document vectors to a query vector.
- **RAG:** retrieves external passages first, then gives those passages to a language model as context.
- **Grounding:** the prompt instructs FLAN-T5 to answer from retrieved context and admit when context is insufficient.
- **Zero-shot screening:** BART-large-MNLI compares the query with candidate labels without training a task-specific classifier on your own labeled safety dataset.

### Important limitations to mention honestly
1. The intent classifier only works well for the label distribution represented in its training data.
2. Retrieval quality depends on the coverage and quality of `cse_topics.csv`.
3. FAISS `IndexFlatIP` performs exact inner-product search; with normalized vectors this is cosine-similarity ranking. It is simple, not approximate nearest-neighbour search.
4. The safety classifier can produce false positives and false negatives. The threshold must be validated on representative examples; it is not a substitute for a dedicated moderation system.
5. The intent is now actually used in the generation prompt. In the original reference notebook, `intent` was passed to `generate_answer` but was not used inside its prompt.
6. The original reference used a `text-generation` pipeline for FLAN-T5. This corrected version uses `text2text-generation`, which is the appropriate pipeline task for T5 models.
